# 阶段 4-2：Q8.8 量化 + 导出 mlp_weights.h

输入：`data/mlp_float.npz`（阶段 4-1 的产物）。
输出：`weights/mlp_weights.h`（阶段 5 的 Vitis HLS 直接 include）。

本 notebook 会做两件**必须做**的检查：

1. **量化前后一致率**（方案书要求 > 97%）。
2. **定点溢出检查**：算出最坏情况下第一层/第二层的累加最大值，
   确认在 int32 里不会溢出。这一步不做，硬件上会出现“训练时对、板上错”的诡异现象。

In [ ]:
import os
import sys
import numpy as np

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
STAGE = os.path.join(ROOT, "stage04_mlp_train")
sys.path.insert(0, ROOT)
sys.path.insert(0, STAGE)

import mlp_common as mc

npz_path = os.path.join(ROOT, "data", "mlp_float.npz")
if not os.path.exists(npz_path):
    raise FileNotFoundError("请先运行 03_mlp_train.ipynb 生成 " + npz_path)

z = np.load(npz_path, allow_pickle=False)
model = {k: z[k] for k in ("w1", "b1", "w2", "b2")}
model["classes"] = int(z["classes"])
print("浮点模型: w1%s b1%s w2%s b2%s" % (model["w1"].shape, model["b1"].shape,
                                          model["w2"].shape, model["b2"].shape))

## 量化 + 一致性 / 溢出检查

In [ ]:
from lib import features_io

csv_path, kind = features_io.pick_features_csv(ROOT)
print("用", os.path.basename(csv_path), "做量化一致性检查")
X, y = features_io.load_csv(csv_path)

qmodel = mc.quantize(model, x_max=X.max(axis=0))
print("溢出/饱和检查:", qmodel["report"])

acc_f = float((mc.predict_float(model, X) == y).mean())
acc_q = float((mc.predict_fixed(qmodel, X) == y).mean())
ag = mc.agreement(qmodel, model, X)
print("浮点准确率 = %.4f" % acc_f)
print("定点准确率 = %.4f" % acc_q)
print("量化前后一致率 = %.4f  (要求 > 0.97)" % ag)
assert ag > 0.97, "量化前后一致率不足，检查 INPUT_SHIFT 或重新训练"

## 导出 C 头文件（阶段 5 的 HLS 会用）

In [ ]:
weights_dir = os.path.join(ROOT, "weights")
os.makedirs(weights_dir, exist_ok=True)
h_path = os.path.join(weights_dir, "mlp_weights.h")

header = mc.to_c_header(qmodel, comment="trained on %s, agreement=%.4f" % (os.path.basename(npz_path), ag))
with open(h_path, "w", encoding="utf-8") as f:
    f.write(header)

print("已写出:", h_path)
print("---- 前 12 行 ----")
print("".join(header.splitlines(True)[:12]))

# 回读一遍，确保文件没写坏（阶段 5 会做同样的事）
back = mc.from_c_header(open(h_path, encoding="utf-8").read())
assert (back["w1"] == qmodel["w1"]).all() and (back["w2"] == qmodel["w2"]).all()
print("头文件回读校验通过")

## 验收

- [x] 一致率 > 97%
- [x] 溢出检查 `int32_ok = True`
- [x] `weights/mlp_weights.h` 生成且能回读

把 `weights/mlp_weights.h` 拷到 `stage05_mlp_hls/src/` 下，进入阶段 5。